In [1]:
# Adaptado de la primera celda de VC_P5_mediapipe.ipynb
# y de la prueba de controles faciales en modo VIDEO.
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import time
import cv2
import mediapipe as mp
from collections import deque, Counter

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode
import numpy as np
from skimage.feature import hog
import joblib


# Clasificadores adicionales
from pathlib import Path
import torch
import torch.nn as nn
from deepface import DeepFace

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CNN:", device)



CNN: cuda


In [2]:
# Cargar el modelo y el normalizador entrenados anteriormente
config = joblib.load("modelo_expresiones.pkl")
modelo = config["modelo"]
scaler = config["scaler"]
class_names = config["class_names"]
width, height = config["width"], config["height"]
ncellsx, ncellsy = config["ncellsx"], config["ncellsy"]

# Valores de la ultima version del notebook de entrenamiento.
# Si se actualizaron los metadatos del pkl, se usan los valores guardados.
orientations = config.get("orientations", 9)
cells_per_block = config.get("cells_per_block", (1, 1))
block_norm = config.get("block_norm", "L1")

# Adaptado de GetHOG en EntrenamientoExpresiones.ipynb
def GetHOG(X, ncellsx, ncellsy, width, height):
    Xhog = []
    for i in range(len(X)):
        gray = X[i].reshape(height, width)
        feat_hog = hog(
            gray, orientations=orientations,
            pixels_per_cell=(height//ncellsy, width//ncellsx),
            cells_per_block=cells_per_block,
            visualize=False, block_norm=block_norm
        )
        Xhog.append(feat_hog)
    return Xhog


def clasificar_svm(rostro):
    gray = cv2.cvtColor(rostro, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (width, height))
    X = [gray.reshape(width * height)]
    Xhog = np.array(GetHOG(X, ncellsx, ncellsy, width, height), dtype="float32")
    # La normalizacion ya fue ajustada durante el entrenamiento.
    Xhog = scaler.transform(Xhog)
    etiqueta = modelo.predict(Xhog)[0]
    return str(class_names[int(etiqueta)])

# Comprobar que la configuracion coincide con el modelo guardado.
prueba = GetHOG([np.zeros(width * height, dtype="uint8")], ncellsx, ncellsy, width, height)
assert len(prueba[0]) == scaler.n_features_in_ == modelo.n_features_in_, "Configuracion HOG incompatible"
print("Modelo cargado correctamente")
print("Clases:", class_names)
print("Caracteristicas HOG:", len(prueba[0]))

# Misma arquitectura de EntrenamientoCNN_Mejorada.ipynb
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.red = nn.Sequential(

            # Primer bloque convolucional
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),

            # Segundo bloque convolucional
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            # Tercer bloque convolucional
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),

            # Clasificacion
            nn.Flatten(),

            nn.Linear(128 * 6 * 6, 128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128, 7)
        )

    def forward(self, x):
        return self.red(x)

# Recuperar los pesos de la CNN ponderada
ruta_cnn = Path("modelo_cnn_ponderado.pth")
if not ruta_cnn.is_file():
    ruta_cnn = Path("modelo_cnn_mejorado.pth")
    print("AVISO: comprueba que este archivo contiene el ultimo entrenamiento ponderado.")

checkpoint = torch.load(ruta_cnn, map_location="cpu", weights_only=True)
modelo_cnn = CNN().to(device)
modelo_cnn.load_state_dict(checkpoint["model_state_dict"])
modelo_cnn.eval()
clases_cnn = checkpoint["class_names"]
print("CNN cargada:", ruta_cnn)


def clasificar_cnn(rostro):
    # Mismo preprocesamiento que en el entrenamiento de la CNN
    gray = cv2.cvtColor(rostro, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (48, 48))
    X = torch.from_numpy(gray).reshape(1, 1, 48, 48).float() / 255
    with torch.inference_mode():
        salida = modelo_cnn(X.to(device))
        etiqueta = salida.argmax(dim=1).item()
    return str(clases_cnn[etiqueta])


# DeepFace usa su modelo preentrenado y su propio preprocesamiento.
DeepFace.build_model(model_name="Emotion", task="facial_attribute")


def clasificar_deepface(rostro):
    resultado = DeepFace.analyze(
        img_path=rostro,
        actions=["emotion"],
        detector_backend="skip",
        enforce_detection=False,
        silent=True
    )
    return resultado[0]["dominant_emotion"]


# Todas las funciones reciben el mismo recorte BGR sin efectos dibujados.
clasificadores = {
    "1": ("HOG + SVM", clasificar_svm),
    "2": ("CNN ponderada", clasificar_cnn),
    "3": ("DeepFace", clasificar_deepface)
}

# Comprobar la inferencia de los tres modelos antes de abrir la camara.
# La imagen artificial solo sirve para inicializar y comprobar las funciones.
rostro_prueba = np.zeros((48, 48, 3), dtype=np.uint8)
for nombre, funcion in clasificadores.values():
    funcion(rostro_prueba)
    print(nombre, "listo")


Modelo cargado correctamente
Clases: ['angry' 'disgust' 'fear' 'happy' 'neutral' 'sad' 'surprise']
Caracteristicas HOG: 144
AVISO: comprueba que este archivo contiene el ultimo entrenamiento ponderado.
CNN cargada: modelo_cnn_mejorado.pth
HOG + SVM listo
CNN ponderada listo
DeepFace listo


In [3]:
# Configuracion del detector, adaptada de VC_P5_mediapipe.ipynb
options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1,
    output_face_blendshapes=False
)

# Obtener el rectangulo del rostro a partir de los landmarks
def limites_rostro(face, frame):
    h, w = frame.shape[:2]
    xs = [lm.x * w for lm in face]
    ys = [lm.y * h for lm in face]
    margen = 0.10 * max(max(xs) - min(xs), max(ys) - min(ys))

    x1 = max(0, int(min(xs) - margen))
    y1 = max(0, int(min(ys) - margen))
    x2 = min(w, int(max(xs) + margen))
    y2 = min(h, int(max(ys) + margen))

    return x1, y1, x2, y2

# Colores BGR para cada expresion
COLORES = {
    "happy": (0, 220, 255),
    "sad": (255, 160, 30),
    "angry": (30, 60, 255),
    "surprise": (0, 240, 255),
    "fear": (220, 90, 190),
    "disgust": (60, 220, 90),
    "neutral": (210, 210, 210)
}


def punto(face, i, frame):
    # Coordenadas normalizadas del landmark -> pixeles
    h, w = frame.shape[:2]
    return int(face[i].x * w), int(face[i].y * h)


def estrella(frame, x, y, radio, color):
    # Estrella de cuatro puntas
    r = max(2, int(radio))
    a = max(1, r // 3)
    pts = np.array([(x, y-r), (x+a, y-a), (x+r, y),
                    (x+a, y+a), (x, y+r), (x-a, y+a),
                    (x-r, y), (x-a, y-a)], dtype=np.int32)
    cv2.fillPoly(frame, [pts], color, lineType=cv2.LINE_AA)


def gota(frame, x, y, r, color):
    r = max(2, int(r))
    cv2.circle(frame, (x, y), r, color, -1, cv2.LINE_AA)
    pts = np.array([(x, y-2*r), (x-r, y), (x+r, y)], dtype=np.int32)
    cv2.fillConvexPoly(frame, pts, color, cv2.LINE_AA)
    cv2.circle(frame, (x-r//3, y-r//3), max(1, r//4), (255, 255, 255), -1)


def aplicar_efecto(frame, face, expresion, ahora):
    if expresion not in COLORES:
        return

    x1, y1, x2, y2 = limites_rostro(face, frame)
    ancho = x2 - x1
    alto = y2 - y1
    cx, cy = (x1+x2)//2, (y1+y2)//2
    # Escala del efecto proporcional al rostro
    s = max(1, min(ancho, alto))
    color = COLORES[expresion]

    # Puntos de referencia: parpados inferiores, frente y ojos
    ojo1 = punto(face, 145, frame)
    ojo2 = punto(face, 374, frame)
    frente = punto(face, 10, frame)

    if expresion == "happy":
        # Estrellas que giran alrededor del rostro
        for i in range(8):
            ang = ahora * 1.2 + i * 2*np.pi/8
            x = int(cx + ancho * 0.60 * np.cos(ang))
            y = int(cy + alto * 0.53 * np.sin(ang))
            r = s * (0.045 + 0.012*np.sin(ahora*5+i))
            estrella(frame, x, y, r, color)

    elif expresion == "sad":
        # Lagrimas animadas desde los parpados inferiores
        for ox, oy in (ojo1, ojo2):
            for i in range(2):
                avance = (ahora * 1.2 + i/2) % 1
                y = int(oy + s*0.05 + avance*s*0.36)
                gota(frame, ox, y, s*0.025, color)

    elif expresion == "angry":
        # Llamas sobre la frente, con una altura variable
        for i in range(-2, 3):
            x = int(frente[0] + i*s*0.14)
            y = int(frente[1] + s*0.035)
            r = max(3, int(s*0.08))
            altura = int(s*(0.20 + 0.04*np.sin(ahora*9+i)))
            pts = np.array([(x-r,y), (x-r,y-altura//3),
                            (x-r//2,y-altura//2), (x,y-altura),
                            (x+r//3,y-altura//3), (x+r,y-altura//2),
                            (x+r,y)], dtype=np.int32)
            cv2.fillPoly(frame, [pts], color, cv2.LINE_AA)
            cv2.ellipse(frame, (x, y-r//2), (max(1,r//2), r),
                        0, 0, 360, (0, 220, 255), -1, cv2.LINE_AA)

    elif expresion == "surprise":
        # Rayos alrededor del rostro y exclamaciones a ambos lados
        for i in range(10):
            ang = i * 2*np.pi/10
            inicio = (int(cx+ancho*0.53*np.cos(ang)),
                      int(cy+alto*0.53*np.sin(ang)))
            fin = (int(cx+ancho*0.66*np.cos(ang)),
                   int(cy+alto*0.66*np.sin(ang)))
            cv2.line(frame, inicio, fin, color, max(2,int(s*0.018)), cv2.LINE_AA)
        for lado in (-1, 1):
            x = int(cx + lado*ancho*0.48)
            y = int(frente[1] - s*0.03)
            cv2.putText(frame, "!", (x,y), cv2.FONT_HERSHEY_SIMPLEX,
                        max(0.5,s/110), color, max(2,int(s/65)), cv2.LINE_AA)

    elif expresion == "fear":
        # Sudor en las sienes y contorno violeta
        cv2.ellipse(frame, (cx,cy), (max(1,ancho//2),max(1,alto//2)),
                    0, 0, 360, color, max(2,int(s*0.015)), cv2.LINE_AA)
        for lado in (-1, 1):
            x = int(cx + lado*ancho*0.42)
            y = int(cy-alto*0.22 + ((ahora*0.9)%1)*s*0.25)
            gota(frame, x, y, s*0.035, (255, 220, 120))

    elif expresion == "disgust":
        # Burbujas verdes que ascienden por los lados
        for lado in (-1, 1):
            for i in range(4):
                avance = (ahora*0.7+i/4) % 1
                x = int(cx+lado*(ancho*0.53+s*0.035*np.sin(ahora*4+i)))
                y = int(y2-avance*alto)
                r = max(2,int(s*(0.025+0.016*avance)))
                cv2.circle(frame, (x,y), r, color, 2, cv2.LINE_AA)
                cv2.circle(frame, (x-r//3,y-r//3), max(1,r//5), (230,255,230), -1)

    elif expresion == "neutral":
        # Gafas que siguen los landmarks de los extremos de los ojos
        a, b = punto(face, 33, frame), punto(face, 133, frame)
        c, d = punto(face, 362, frame), punto(face, 263, frame)
        centros = [((a[0]+b[0])//2,(a[1]+b[1])//2),
                   ((c[0]+d[0])//2,(c[1]+d[1])//2)]
        cv2.line(frame, centros[0], centros[1], (25,25,25), max(2,int(s*0.035)), cv2.LINE_AA)
        for centro in centros:
            rx, ry = max(2,int(s*0.18)), max(2,int(s*0.075))
            cv2.ellipse(frame, centro, (rx,ry), 0, 0, 360, (25,25,25), -1, cv2.LINE_AA)
            cv2.ellipse(frame, centro, (rx,ry), 0, 0, 360, (160,160,160), 2, cv2.LINE_AA)
            x,y = centro
            cv2.line(frame, (x-rx//2,y+ry//3), (x,y-ry//2), (130,130,130), 2, cv2.LINE_AA)


In [4]:
# Captura de camara y suavizado temporal de las predicciones
INTERVALO = 0.25  # Segundos entre clasificaciones
historial = deque(maxlen=5)
expresion_estable = None
ultimo_analisis = float('-inf')
ultimo_timestamp = -1
modelo_activo = "1"
tiempo_inferencia = None
intervalos_frames = deque(maxlen=30)
ultimo_frame = None

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)
ventana = "Face Effects"
mostrar_landmarks = False

try:
    if not cap.isOpened():
        raise RuntimeError("No se pudo abrir la camara")

    with FaceLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

            ahora = time.monotonic()
            if ultimo_frame is not None:
                intervalos_frames.append(ahora - ultimo_frame)
            ultimo_frame = ahora
            timestamp = max(int(ahora * 1000), ultimo_timestamp + 1)
            ultimo_timestamp = timestamp
            result = landmarker.detect_for_video(mp_image, timestamp)

            texto = "Rostro no detectado"
            rostro_valido = False

            if result.face_landmarks:
                face = result.face_landmarks[0]
                h, w = frame.shape[:2]
                x1, y1, x2, y2 = limites_rostro(face, frame)

                if x2 > x1 and y2 > y1:
                    rostro_valido = True

                    # Clasificar como maximo una vez cada 0.25 segundos
                    if ahora - ultimo_analisis >= INTERVALO:
                        rostro = frame[y1:y2, x1:x2]
                        nombre, clasificar = clasificadores[modelo_activo]
                        t0 = time.perf_counter()
                        prediccion = clasificar(rostro)
                        tiempo_inferencia = (time.perf_counter() - t0) * 1000
                        historial.append(prediccion)
                        ultimo_analisis = ahora

                        # Cambiar la expresion solo si tiene al menos 3 votos
                        candidata, votos = Counter(historial).most_common(1)[0]
                        if votos >= 3:
                            expresion_estable = candidata

                    texto = ("Expresion: " + expresion_estable
                             if expresion_estable else "Analizando...")

                    # Dibujar despues de clasificar para no alterar HOG
                    if expresion_estable:
                        aplicar_efecto(frame, face, expresion_estable, ahora)

                    # Landmarks opcionales para mostrar el funcionamiento
                    if mostrar_landmarks:
                        cv2.rectangle(frame, (x1, y1), (x2-1, y2-1), (0, 255, 0), 2)
                        for lm in face:
                            cv2.circle(frame, (int(lm.x * w), int(lm.y * h)),
                                       1, (0, 255, 0), -1)

            # Reiniciar el historial si se pierde el rostro
            if not rostro_valido:
                historial.clear()
                expresion_estable = None
                ultimo_analisis = float('-inf')
                tiempo_inferencia = None

            color = COLORES.get(expresion_estable, (255, 255, 255))
            cv2.putText(frame, texto, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (0, 0, 0), 5, cv2.LINE_AA)
            cv2.putText(frame, texto, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, color, 2, cv2.LINE_AA)
            # Informacion comun a los tres clasificadores
            fps = len(intervalos_frames) / sum(intervalos_frames) if intervalos_frames and sum(intervalos_frames) > 0 else 0
            nombre = clasificadores[modelo_activo][0]
            ms = f"{tiempo_inferencia:.1f} ms" if tiempo_inferencia is not None else "--"
            info = f"{nombre} | Inferencia: {ms} | FPS: {fps:.1f}"
            cv2.putText(frame, info, (10, 58), cv2.FONT_HERSHEY_SIMPLEX,
                        0.55, (0, 0, 0), 4, cv2.LINE_AA)
            cv2.putText(frame, info, (10, 58), cv2.FONT_HERSHEY_SIMPLEX,
                        0.55, (255, 255, 255), 1, cv2.LINE_AA)
            ayuda = "1: SVM   2: CNN   3: DeepFace   L: puntos   ESC: salir"
            pos = (10, frame.shape[0] - 12)
            cv2.putText(frame, ayuda, pos, cv2.FONT_HERSHEY_SIMPLEX,
                        0.48, (0, 0, 0), 4, cv2.LINE_AA)
            cv2.putText(frame, ayuda, pos, cv2.FONT_HERSHEY_SIMPLEX,
                        0.48, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.imshow(ventana, frame)

            tecla = cv2.waitKey(1) & 0xFF
            if tecla == 27:
                break
            if tecla in (ord('1'), ord('2'), ord('3')):
                nuevo = chr(tecla)
                if nuevo != modelo_activo:
                    modelo_activo = nuevo
                    historial.clear()
                    expresion_estable = None
                    ultimo_analisis = float('-inf')
                    tiempo_inferencia = None
                    intervalos_frames.clear()
                    ultimo_frame = None
            if tecla in (ord('l'), ord('L')):
                mostrar_landmarks = not mostrar_landmarks
            try:
                if cv2.getWindowProperty(ventana, cv2.WND_PROP_VISIBLE) < 1:
                    break
            except cv2.error:
                break

except KeyboardInterrupt:
    print("Aplicacion interrumpida")
finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Camara liberada")234232312

Camara liberada
